# Multimodal FER Notebook

Notebook này huấn luyện mô hình:

**Face RGB encoder + UpperBody RGB encoder + 478-point FaceMesh encoder → Region-aware cross attention → Confidence-gated multimodal fusion → CE + SupCon**

Dữ liệu:
- Face images: `/content/drive/MyDrive/KhoaLuan/ThaoStudentEmoData/FinalData/Face`
- UpperBody images: `/content/drive/MyDrive/KhoaLuan/ThaoStudentEmoData/FinalData/UpperBody`
- FaceMesh JSON: `/content/drive/MyDrive/KhoaLuan/ThaoStudentEmoData/FinalData/facemesh.json`
- Train CSV: `/content/drive/MyDrive/KhoaLuan/ThaoStudentEmoData/FinalData/train.csv`
- Valid CSV: `/content/drive/MyDrive/KhoaLuan/ThaoStudentEmoData/FinalData/valid.csv`

CSV có 2 cột:
- `filename`
- `true_label_no`

Labels:
- `0`: enjoyment
- `1`: confusion
- `2`: neutrality
- `3`: fatigue
- `4`: distraction

In [ ]:
# Nếu chạy trên Colab:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import os
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

In [3]:
# =========================
# Config
# =========================

@dataclass
class CFG:
    seed: int = 42

    face_dir: str = "/content/drive/MyDrive/ThaoStudentEmoData/FinalData/Face"
    body_dir: str = "/content/drive/MyDrive/ThaoStudentEmoData/FinalData/UpperBody"
    facemesh_json: str = "/content/drive/MyDrive/ThaoStudentEmoData/FinalData/facemesh.json"
    train_csv: str = "/content/drive/MyDrive/ThaoStudentEmoData/FinalData/train.csv"
    valid_csv: str = "/content/drive/MyDrive/ThaoStudentEmoData/FinalData/valid.csv"

    face_size: int = 224
    body_size: int = 224
    mesh_points: int = 478
    mesh_in_dim: int = 4    # x,y,z,conf

    num_classes: int = 5
    class_names: tuple = ("enjoyment", "confusion", "neutrality", "fatigue", "distraction")

    batch_size: int = 16
    num_workers: int = 2
    epochs: int = 20
    early_stop_patience: int = 10  # dừng nếu valid accuracy không cải thiện trong 10 epoch
    monitor_metric: str = "valid_acc"

    lr: float = 3e-4
    weight_decay: float = 1e-4

    face_embed_dim: int = 256
    body_embed_dim: int = 256
    mesh_embed_dim: int = 256
    fusion_dim: int = 256
    num_region_tokens: int = 8
    num_heads: int = 8
    dropout: float = 0.1

    ce_weight: float = 1.0
    supcon_weight: float = 0.15
    label_smoothing: float = 0.05
    temperature: float = 0.07

    save_dir: str = "/content/drive/MyDrive/ThaoStudentEmoData/checkpoints_multimodal_fer"
    cm_png_name: str = "confusion_matrix_valid_percent.png"
    cm_npy_name: str = "confusion_matrix_valid_percent.npy"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CFG()
os.makedirs(cfg.save_dir, exist_ok=True)
cfg


CFG(seed=42, face_dir='/content/drive/MyDrive/ThaoStudentEmoData/FinalData/Face', body_dir='/content/drive/MyDrive/ThaoStudentEmoData/FinalData/UpperBody', facemesh_json='/content/drive/MyDrive/ThaoStudentEmoData/FinalData/facemesh.json', train_csv='/content/drive/MyDrive/ThaoStudentEmoData/FinalData/train.csv', valid_csv='/content/drive/MyDrive/ThaoStudentEmoData/FinalData/valid.csv', face_size=224, body_size=224, mesh_points=478, mesh_in_dim=4, num_classes=5, class_names=('enjoyment', 'confusion', 'neutrality', 'fatigue', 'distraction'), batch_size=16, num_workers=2, epochs=20, lr=0.0003, weight_decay=0.0001, face_embed_dim=256, body_embed_dim=256, mesh_embed_dim=256, fusion_dim=256, num_region_tokens=8, num_heads=8, dropout=0.1, ce_weight=1.0, supcon_weight=0.15, label_smoothing=0.05, temperature=0.07, save_dir='/content/drive/MyDrive/ThaoStudentEmoData/checkpoints_multimodal_fer', device='cpu')

In [4]:
# =========================
# Utils
# =========================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)

def pil_loader(path):
    with open(path, "rb") as f:
        img = Image.open(f)
        return img.convert("RGB")

def resolve_img_path(root_dir, filename):
    root_dir = Path(root_dir)
    filename = str(filename).strip()

    candidates = [root_dir / filename]

    stem = Path(filename).stem
    suffix = Path(filename).suffix.lower()

    if suffix:
        exts = [suffix]
    else:
        exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

    for ext in exts + [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        candidates.append(root_dir / f"{stem}{ext}")

    seen = set()
    uniq = []
    for p in candidates:
        ps = str(p)
        if ps not in seen:
            uniq.append(p)
            seen.add(ps)

    for p in uniq:
        if p.exists():
            return str(p)

    raise FileNotFoundError(f"Không tìm thấy ảnh cho {filename} trong {root_dir}")

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [5]:
# =========================
# Load FaceMesh JSON
# =========================

def load_facemesh_map(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    facemesh_map = {}
    skipped = 0

    for fname, item in raw.items():
        if not isinstance(item, dict):
            skipped += 1
            continue
        if "w" not in item or "h" not in item or "landmarks" not in item:
            skipped += 1
            continue

        pts = np.asarray(item["landmarks"], dtype=np.float32)
        if pts.shape != (cfg.mesh_points, 3):
            skipped += 1
            continue

        facemesh_map[str(fname).strip()] = {
            "w": float(item["w"]),
            "h": float(item["h"]),
            "landmarks": item["landmarks"],
        }

    print(f"Loaded facemesh: {len(facemesh_map)} samples | skipped: {skipped}")
    return facemesh_map

facemesh_map = load_facemesh_map(cfg.facemesh_json)

Loaded facemesh: 10163 samples | skipped: 0


In [6]:
# =========================
# Transforms
# =========================

train_tfms = transforms.Compose([
    transforms.Resize((cfg.face_size, cfg.face_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

valid_tfms = transforms.Compose([
    transforms.Resize((cfg.face_size, cfg.face_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

body_train_tfms = transforms.Compose([
    transforms.Resize((cfg.body_size, cfg.body_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

body_valid_tfms = transforms.Compose([
    transforms.Resize((cfg.body_size, cfg.body_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [7]:
# =========================
# Dataset
# =========================

class StudentEmoDataset(Dataset):
    def __init__(self, csv_path, face_dir, body_dir, facemesh_map, face_tfm, body_tfm):
        self.df = pd.read_csv(csv_path)
        assert "filename" in self.df.columns and "true_label_no" in self.df.columns

        self.face_dir = face_dir
        self.body_dir = body_dir
        self.facemesh_map = facemesh_map
        self.face_tfm = face_tfm
        self.body_tfm = body_tfm

        keep_rows = []
        missing = 0

        for _, row in self.df.iterrows():
            fname = str(row["filename"]).strip()
            base = os.path.splitext(fname)[0]

            has_mesh = (fname in self.facemesh_map) or (base in self.facemesh_map)

            try:
                _ = resolve_img_path(self.face_dir, fname)
                _ = resolve_img_path(self.body_dir, fname)
                has_imgs = True
            except FileNotFoundError:
                has_imgs = False

            if has_mesh and has_imgs:
                keep_rows.append(row)
            else:
                missing += 1

        self.df = pd.DataFrame(keep_rows).reset_index(drop=True)
        print(f"{os.path.basename(csv_path)}: giữ {len(self.df)} samples, bỏ {missing} samples thiếu dữ liệu")

    def __len__(self):
        return len(self.df)

    def _get_mesh_item(self, filename):
        base = os.path.splitext(filename)[0]
        if filename in self.facemesh_map:
            return self.facemesh_map[filename]
        if base in self.facemesh_map:
            return self.facemesh_map[base]
        raise KeyError(f"Không tìm thấy facemesh cho: {filename}")

    def _build_mesh_feature(self, filename):
        item = self._get_mesh_item(filename)

        w = float(item["w"])
        h = float(item["h"])
        landmarks = np.asarray(item["landmarks"], dtype=np.float32)  # (478, 3)

        if landmarks.shape != (cfg.mesh_points, 3):
            raise ValueError(f"{filename}: landmarks shape phải là ({cfg.mesh_points},3), nhưng nhận {landmarks.shape}")

        coords = landmarks.copy()
        coords[:, 0] = coords[:, 0] / max(w, 1.0)
        coords[:, 1] = coords[:, 1] / max(h, 1.0)
        coords[:, 2] = coords[:, 2] / max(max(w, h), 1.0)

        center = coords.mean(axis=0, keepdims=True)
        coords_centered = coords - center
        scale = np.linalg.norm(coords_centered[:, :2], axis=1).max() + 1e-6
        coords_norm = coords_centered / scale

        conf = np.ones((cfg.mesh_points, 1), dtype=np.float32)
        mesh_feat = np.concatenate([coords_norm, conf], axis=1).astype(np.float32)  # (478,4)
        return mesh_feat

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = str(row["filename"]).strip()
        label = int(row["true_label_no"])

        face_path = resolve_img_path(self.face_dir, filename)
        body_path = resolve_img_path(self.body_dir, filename)

        face_img = pil_loader(face_path)
        body_img = pil_loader(body_path)

        face_tensor = self.face_tfm(face_img)
        body_tensor = self.body_tfm(body_img)
        mesh_feat = self._build_mesh_feature(filename)

        return {
            "face": face_tensor,
            "body": body_tensor,
            "mesh": torch.tensor(mesh_feat, dtype=torch.float32),
            "label": torch.tensor(label, dtype=torch.long),
            "filename": filename,
        }

train_ds = StudentEmoDataset(cfg.train_csv, cfg.face_dir, cfg.body_dir, facemesh_map, train_tfms, body_train_tfms)
valid_ds = StudentEmoDataset(cfg.valid_csv, cfg.face_dir, cfg.body_dir, facemesh_map, valid_tfms, body_valid_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
)

print("train:", len(train_ds), "valid:", len(valid_ds))

train.csv: giữ 8016 samples, bỏ 0 samples thiếu dữ liệu
valid.csv: giữ 2076 samples, bỏ 0 samples thiếu dữ liệu
train: 8016 valid: 2076


In [8]:
# Kiểm tra 1 batch
batch = next(iter(train_loader))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), len(v))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


face (16, 3, 224, 224) torch.float32
body (16, 3, 224, 224) torch.float32
mesh (16, 478, 4) torch.float32
label (16,) torch.int64
filename <class 'list'> 16


In [9]:
# =========================
# Model blocks
# =========================

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class CNNTokenEncoder(nn.Module):
    def __init__(self, model_name="resnet18", out_dim=256, pretrained=True):
        super().__init__()
        if model_name == "resnet18":
            weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            backbone = models.resnet18(weights=weights)
            self.feature_dim = 512
        elif model_name == "resnet34":
            weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
            backbone = models.resnet34(weights=weights)
            self.feature_dim = 512
        else:
            raise ValueError(f"Unsupported model_name: {model_name}")

        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.proj = nn.Conv2d(self.feature_dim, out_dim, kernel_size=1)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)              # (B, C, H, W)
        x = self.proj(x)                # (B, D, H, W)
        B, D, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)   # (B, HW, D)
        global_token = x.mean(dim=(2, 3))       # (B, D)
        return tokens, global_token


class MeshGraphEncoder(nn.Module):
    def __init__(self, in_dim=4, hidden_dim=256, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=8,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, mesh):
        # mesh: (B, 478, 4)
        x = self.input_proj(mesh)
        x = self.encoder(x)
        x = self.norm(x)
        global_token = x.mean(dim=1)
        return x, global_token


class RegionAwareCrossAttention(nn.Module):
    def __init__(self, dim=256, num_region_tokens=8, num_heads=8, dropout=0.1):
        super().__init__()
        self.region_tokens = nn.Parameter(torch.randn(1, num_region_tokens, dim) * 0.02)

        self.mesh_to_face = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.mesh_to_body = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

        self.face_norm = nn.LayerNorm(dim)
        self.body_norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, mesh_tokens, face_tokens, body_tokens):
        B = mesh_tokens.size(0)
        region_tokens = self.region_tokens.expand(B, -1, -1)  # (B, R, D)

        mesh_summary = mesh_tokens[:, :region_tokens.size(1), :]
        query = region_tokens + mesh_summary

        face_out, _ = self.mesh_to_face(query, face_tokens, face_tokens)
        body_out, _ = self.mesh_to_body(query, body_tokens, body_tokens)

        face_out = self.face_norm(face_out + query)
        body_out = self.body_norm(body_out + query)

        fused_regions = 0.5 * (face_out + body_out)
        fused_regions = fused_regions + self.ffn(fused_regions)
        return fused_regions


class ConfidenceGatedFusion(nn.Module):
    def __init__(self, dim=256, dropout=0.1):
        super().__init__()
        self.face_gate = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim // 2, 1),
        )
        self.body_gate = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim // 2, 1),
        )
        self.mesh_gate = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim // 2, 1),
        )

    def forward(self, face_feat, body_feat, mesh_feat):
        logits = torch.cat([
            self.face_gate(face_feat),
            self.body_gate(body_feat),
            self.mesh_gate(mesh_feat),
        ], dim=1)                        # (B, 3)
        weights = torch.softmax(logits, dim=1)

        fused = (
            weights[:, 0:1] * face_feat +
            weights[:, 1:2] * body_feat +
            weights[:, 2:3] * mesh_feat
        )
        return fused, weights

In [10]:
# =========================
# Full model
# =========================

class MultimodalFERModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.face_encoder = CNNTokenEncoder("resnet18", out_dim=cfg.face_embed_dim, pretrained=True)
        self.body_encoder = CNNTokenEncoder("resnet18", out_dim=cfg.body_embed_dim, pretrained=True)
        self.mesh_encoder = MeshGraphEncoder(
            in_dim=cfg.mesh_in_dim,
            hidden_dim=cfg.mesh_embed_dim,
            num_layers=4,
            dropout=cfg.dropout,
        )

        self.region_cross_attn = RegionAwareCrossAttention(
            dim=cfg.fusion_dim,
            num_region_tokens=cfg.num_region_tokens,
            num_heads=cfg.num_heads,
            dropout=cfg.dropout,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.fusion_dim,
            nhead=cfg.num_heads,
            dim_feedforward=cfg.fusion_dim * 4,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.cls_token = nn.Parameter(torch.randn(1, 1, cfg.fusion_dim) * 0.02)

        self.gated_fusion = ConfidenceGatedFusion(cfg.fusion_dim, dropout=cfg.dropout)

        self.pre_logits = nn.Sequential(
            nn.LayerNorm(cfg.fusion_dim),
            nn.Linear(cfg.fusion_dim, cfg.fusion_dim),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
        )

        self.classifier = nn.Linear(cfg.fusion_dim, cfg.num_classes)

        self.projection_head = nn.Sequential(
            nn.LayerNorm(cfg.fusion_dim),
            nn.Linear(cfg.fusion_dim, cfg.fusion_dim),
            nn.GELU(),
            nn.Linear(cfg.fusion_dim, 128),
        )

    def forward(self, face, body, mesh):
        face_tokens, face_global = self.face_encoder(face)       # (B,Nf,D), (B,D)
        body_tokens, body_global = self.body_encoder(body)       # (B,Nb,D), (B,D)
        mesh_tokens, mesh_global = self.mesh_encoder(mesh)       # (B,478,D), (B,D)

        region_tokens = self.region_cross_attn(mesh_tokens, face_tokens, body_tokens)  # (B,R,D)

        B = face.size(0)
        cls_token = self.cls_token.expand(B, -1, -1)

        fusion_tokens = torch.cat([
            cls_token,
            region_tokens,
            face_global.unsqueeze(1),
            body_global.unsqueeze(1),
            mesh_global.unsqueeze(1),
        ], dim=1)

        fusion_tokens = self.fusion_transformer(fusion_tokens)
        cls_feat = fusion_tokens[:, 0]
        face_feat = fusion_tokens[:, -3]
        body_feat = fusion_tokens[:, -2]
        mesh_feat = fusion_tokens[:, -1]

        gated_feat, gate_weights = self.gated_fusion(face_feat, body_feat, mesh_feat)
        final_feat = cls_feat + gated_feat
        final_feat = self.pre_logits(final_feat)

        logits = self.classifier(final_feat)
        proj = F.normalize(self.projection_head(final_feat), dim=1)

        return {
            "logits": logits,
            "embedding": final_feat,
            "proj": proj,
            "gate_weights": gate_weights,
            "face_feat": face_feat,
            "body_feat": body_feat,
            "mesh_feat": mesh_feat,
        }

model = MultimodalFERModel(cfg).to(cfg.device)
print(model.__class__.__name__)
print("Trainable params:", f"{count_parameters(model):,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 153MB/s]


MultimodalFERModel
Trainable params: 28,679,176


/tmp/ipykernel_4275/1640434814.py:73: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
/tmp/ipykernel_4275/13771273.py:33: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)


In [11]:
# =========================
# Losses
# =========================

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        # features: (B, D), already normalized
        # labels: (B,)
        device = features.device
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        logits = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(logits, dim=1, keepdim=True)
        logits = logits - logits_max.detach()

        logits_mask = torch.ones_like(mask) - torch.eye(mask.shape[0], device=device)
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)

        mask_sum = mask.sum(dim=1)
        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (mask_sum + 1e-12)

        loss = -mean_log_prob_pos
        valid = (mask_sum > 0).float()
        loss = (loss * valid).sum() / (valid.sum() + 1e-12)
        return loss

ce_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
supcon_criterion = SupConLoss(temperature=cfg.temperature)

In [12]:
# =========================
# Optimizer / Scheduler
# =========================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.epochs,
    eta_min=1e-6,
)

In [13]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix_percent(y_true, y_pred, class_names, save_path=None, return_matrix=False):
    """
    Vẽ confusion matrix chuẩn hóa theo hàng (mỗi hàng = 100%).

    Args:
        y_true: ground-truth labels
        y_pred: predicted labels
        class_names: list/tupple tên lớp
        save_path: nếu có thì lưu PNG
        return_matrix: trả về ma trận % nếu True
    """
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_percent = cm.astype(np.float32) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    cm_percent = cm_percent * 100.0

    annot = np.empty_like(cm_percent).astype(object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm_percent[i, j]:.2f}%"

    plt.figure(figsize=(8, 6))
    ax = sns.heatmap(
        cm_percent,
        annot=annot,
        fmt="",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=True,
        vmin=0,
        vmax=100,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Validation Confusion Matrix (%)")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Saved confusion matrix -> {save_path}")

    plt.show()

    if return_matrix:
        return cm_percent


In [14]:
# =========================
# Train / Eval
# =========================

def compute_loss(outputs, labels):
    logits = outputs["logits"]
    proj = outputs["proj"]

    ce_loss = ce_criterion(logits, labels)

    if labels.unique().numel() > 1 and labels.numel() > 1:
        supcon_loss = supcon_criterion(proj, labels)
    else:
        supcon_loss = torch.tensor(0.0, device=labels.device)

    total_loss = cfg.ce_weight * ce_loss + cfg.supcon_weight * supcon_loss
    return total_loss, {
        "ce_loss": ce_loss.detach().item(),
        "supcon_loss": supcon_loss.detach().item(),
        "total_loss": total_loss.detach().item(),
    }


def train_one_epoch(model, loader, optimizer, device):
    model.train()

    losses = []
    preds_all = []
    labels_all = []

    pbar = tqdm(loader, desc="Train", leave=False)
    for batch in pbar:
        face = batch["face"].to(device, non_blocking=True)
        body = batch["body"].to(device, non_blocking=True)
        mesh = batch["mesh"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(face, body, mesh)
        loss, loss_dict = compute_loss(outputs, labels)
        loss.backward()
        optimizer.step()

        logits = outputs["logits"]
        preds = logits.argmax(dim=1)

        losses.append(loss_dict["total_loss"])
        preds_all.extend(preds.detach().cpu().numpy().tolist())
        labels_all.extend(labels.detach().cpu().numpy().tolist())

        pbar.set_postfix(
            loss=f"{np.mean(losses):.4f}",
            acc=f"{accuracy_score(labels_all, preds_all):.4f}",
        )

    metrics = {
        "loss": float(np.mean(losses)),
        "acc": accuracy_score(labels_all, preds_all),
        "f1_macro": f1_score(labels_all, preds_all, average="macro"),
    }
    return metrics


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    losses = []
    preds_all = []
    labels_all = []
    filenames_all = []
    gate_weights_all = []

    pbar = tqdm(loader, desc="Valid", leave=False)
    for batch in pbar:
        face = batch["face"].to(device, non_blocking=True)
        body = batch["body"].to(device, non_blocking=True)
        mesh = batch["mesh"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(face, body, mesh)
        loss, loss_dict = compute_loss(outputs, labels)

        logits = outputs["logits"]
        preds = logits.argmax(dim=1)

        losses.append(loss_dict["total_loss"])
        preds_all.extend(preds.cpu().numpy().tolist())
        labels_all.extend(labels.cpu().numpy().tolist())
        filenames_all.extend(batch["filename"])
        gate_weights_all.append(outputs["gate_weights"].cpu())

    gate_weights_all = torch.cat(gate_weights_all, dim=0)

    metrics = {
        "loss": float(np.mean(losses)),
        "acc": accuracy_score(labels_all, preds_all),
        "f1_macro": f1_score(labels_all, preds_all, average="macro"),
        "preds": preds_all,
        "labels": labels_all,
        "filenames": filenames_all,
        "gate_weights": gate_weights_all,
    }
    return metrics


def save_checkpoint(model, optimizer, scheduler, epoch, best_score, path):
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": epoch,
        "best_score": best_score,
        "cfg": cfg.__dict__,
    }, path)

In [15]:
# =========================
# Main training loop
# =========================

best_score = -1.0
best_epoch = 0
epochs_without_improve = 0
history = []

for epoch in range(1, cfg.epochs + 1):
    print(f"\nEpoch [{epoch}/{cfg.epochs}]")

    train_metrics = train_one_epoch(model, train_loader, optimizer, cfg.device)
    valid_metrics = evaluate(model, valid_loader, cfg.device)

    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_f1_macro": train_metrics["f1_macro"],
        "valid_loss": valid_metrics["loss"],
        "valid_acc": valid_metrics["acc"],
        "valid_f1_macro": valid_metrics["f1_macro"],
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)

    print(pd.DataFrame([row]))

    score = valid_metrics["acc"]  # early stop theo accuracy
    if score > best_score:
        best_score = score
        best_epoch = epoch
        epochs_without_improve = 0
        ckpt_path = os.path.join(cfg.save_dir, "best_multimodal_fer.pth")
        save_checkpoint(model, optimizer, scheduler, epoch, best_score, ckpt_path)
        print(f"Saved best checkpoint -> {ckpt_path}")
        print(f"New best valid accuracy: {best_score * 100:.2f}%")
    else:
        epochs_without_improve += 1
        print(
            f"No improvement in valid accuracy for {epochs_without_improve}/{cfg.early_stop_patience} epoch(s). "
            f"Best: {best_score * 100:.2f}% at epoch {best_epoch}"
        )

    if epochs_without_improve >= cfg.early_stop_patience:
        print(f"Early stopping triggered at epoch {epoch}. Best valid accuracy: {best_score * 100:.2f}%")
        break

history_df = pd.DataFrame(history)
history_df



Epoch [1/20]


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Train:   0%|          | 0/501 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# =========================
# Load best checkpoint
# =========================

best_ckpt = os.path.join(cfg.save_dir, "best_multimodal_fer.pth")
ckpt = torch.load(best_ckpt, map_location=cfg.device)
model.load_state_dict(ckpt["model"])
print("Loaded:", best_ckpt)
print("Best valid f1_macro:", ckpt["best_score"])

In [ ]:
# =========================
# Final evaluation
# =========================

valid_metrics = evaluate(model, valid_loader, cfg.device)

print("Valid loss:", valid_metrics["loss"])
print("Valid acc:", valid_metrics["acc"])
print("Valid f1_macro:", valid_metrics["f1_macro"])

cm_png_path = os.path.join(cfg.save_dir, cfg.cm_png_name)
cm_npy_path = os.path.join(cfg.save_dir, cfg.cm_npy_name)

cm_percent = plot_confusion_matrix_percent(
    valid_metrics["labels"],
    valid_metrics["preds"],
    cfg.class_names,
    save_path=cm_png_path,
    return_matrix=True,
)

np.save(cm_npy_path, cm_percent)
print(f"Saved confusion matrix array -> {cm_npy_path}")


In [ ]:
# =========================
# Gate weight analysis
# =========================

gate_weights = valid_metrics["gate_weights"].numpy()
gate_df = pd.DataFrame(gate_weights, columns=["face_gate", "body_gate", "mesh_gate"])
gate_df.describe()

In [ ]:
# Trung bình gate theo toàn bộ tập valid
gate_df.mean()

In [ ]:
# =========================
# Inference helper
# =========================

@torch.no_grad()
def predict_one(model, face_path, body_path, mesh_item, device):
    model.eval()

    face_img = pil_loader(face_path)
    body_img = pil_loader(body_path)

    face_tensor = valid_tfms(face_img).unsqueeze(0).to(device)
    body_tensor = body_valid_tfms(body_img).unsqueeze(0).to(device)

    w = float(mesh_item["w"])
    h = float(mesh_item["h"])
    landmarks = np.asarray(mesh_item["landmarks"], dtype=np.float32)

    coords = landmarks.copy()
    coords[:, 0] = coords[:, 0] / max(w, 1.0)
    coords[:, 1] = coords[:, 1] / max(h, 1.0)
    coords[:, 2] = coords[:, 2] / max(max(w, h), 1.0)

    center = coords.mean(axis=0, keepdims=True)
    coords_centered = coords - center
    scale = np.linalg.norm(coords_centered[:, :2], axis=1).max() + 1e-6
    coords_norm = coords_centered / scale
    conf = np.ones((cfg.mesh_points, 1), dtype=np.float32)
    mesh_feat = np.concatenate([coords_norm, conf], axis=1).astype(np.float32)

    mesh_tensor = torch.tensor(mesh_feat, dtype=torch.float32).unsqueeze(0).to(device)

    outputs = model(face_tensor, body_tensor, mesh_tensor)
    probs = torch.softmax(outputs["logits"], dim=1)[0].cpu().numpy()
    pred = int(np.argmax(probs))
    gates = outputs["gate_weights"][0].cpu().numpy()

    return {
        "pred_idx": pred,
        "pred_name": cfg.class_names[pred],
        "probs": {cfg.class_names[i]: float(probs[i]) for i in range(cfg.num_classes)},
        "gates": {
            "face": float(gates[0]),
            "body": float(gates[1]),
            "mesh": float(gates[2]),
        }
    }

In [ ]:
# Ví dụ inference với 1 mẫu trong valid.csv
valid_df = pd.read_csv(cfg.valid_csv)
sample_fname = str(valid_df.iloc[0]["filename"]).strip()

face_path = resolve_img_path(cfg.face_dir, sample_fname)
body_path = resolve_img_path(cfg.body_dir, sample_fname)
mesh_item = facemesh_map[sample_fname] if sample_fname in facemesh_map else facemesh_map[Path(sample_fname).stem]

result = predict_one(model, face_path, body_path, mesh_item, cfg.device)
result

In [ ]:
# =========================
# Export history / predictions
# =========================

history_path = os.path.join(cfg.save_dir, "history.csv")
history_df.to_csv(history_path, index=False)

pred_df = pd.DataFrame({
    "filename": valid_metrics["filenames"],
    "true_label": valid_metrics["labels"],
    "pred_label": valid_metrics["preds"],
})
pred_path = os.path.join(cfg.save_dir, "valid_predictions.csv")
pred_df.to_csv(pred_path, index=False)

print("Saved history:", history_path)
print("Saved predictions:", pred_path)

## Ghi chú

1. Notebook này dùng backbone `resnet18` để dễ chạy trên Colab.
2. Nếu muốn mạnh hơn, có thể đổi sang:
   - `resnet34`
   - EfficientNet
   - Swin / ConvNeXt
3. `RegionAwareCrossAttention` hiện dùng **learnable region tokens** + mesh summary.
4. `ConfidenceGatedFusion` trả về trọng số cho 3 modality:
   - face
   - body
   - mesh
5. Loss:
   - `CrossEntropyLoss(label_smoothing=0.05)`
   - `SupConLoss`